In [21]:
import os
import json
import zipfile
import numpy as np
import copy
import requests

import pandas as pd
import geopandas as gpd

import shapely.wkt as wkt
from shapely.geometry import LineString
from shapely.geometry import Polygon
from shapely.geometry import Point

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import BoundaryNorm

from math import radians, cos, sin, asin, sqrt

In [22]:
DIR="/home/dy23a.fsu/st/datasets/raw"
os.makedirs(DIR, exist_ok=True)
DIR=os.path.join(DIR,"Chicago")
os.makedirs(DIR, exist_ok=True)

Chicago_Data_Portal_KEY_ID = "6kulio3y2buy37oz16yzq7y0v"
Chicago_Data_Portal_KEY_SECRET = "4s77fm76dc4n4jqn0s3zba56qjjd2an9fp3gf32aiatsln3n7h"

chicago_shape="./Chicago Community Areas.geojson"
chi_shapefile_url="https://data.cityofchicago.org/api/v3/views/igwz-8jzy/query.geojson"

taxi_orig_file=os.path.join(DIR,"chi_taxi_2025_15min.csv")
TNP_orig_file=os.path.join(DIR,"chi_TNP_2025_15min.csv")
bike_orig_dir=os.path.join(DIR,"bike")
scooter_orig_file=os.path.join(DIR,"chi_scooter_2025_60min.csv")

# Download
Community Areas https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Community-Areas-Map/cauq-8yn6  
Taxi 2025 15min https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data    
Transportation Network Providers 2025 15min https://data.cityofchicago.org/Transportation/Transportation-Network-Providers-Trips-2025-/6dvr-xwnh/about_data  
Bike 2025 15min https://data.cityofchicago.org/Transportation/Divvy-Trips/fg6s-gzvg/about_data  
Scooter 2025 1hour https://data.cityofchicago.org/Transportation/E-Scooter-Trips/2i5w-ykuw/about_data  

## 1 Shapefile

In [23]:
if not os.path.exists(chicago_shape):
    response = requests.get(chi_shapefile_url)
    with open(chicago_shape, "w", encoding="utf-8") as file:
        file.write(response.text)

## 2 Taxi

In [24]:
def download_from_api(API_URL, base_query, file, page_size=50000):
    offset = 0
    total = 0
    with open(file, mode="w", encoding="utf-8") as f:
        header_written = False
        while True:
            query = dict(base_query)
            query["$limit"] = page_size
            query["$offset"] = offset
            response = requests.get(
                API_URL,
                auth=(Chicago_Data_Portal_KEY_ID, Chicago_Data_Portal_KEY_SECRET),
                params=query,
                timeout=300,
            )
            if response.status_code != 200:
                print(f"Error: {response.status_code}")
                print(f"Info: {response.text}")
                return
            lines = response.text.splitlines()
            if len(lines) <= 1:
                break
            header, data_lines = lines[0], lines[1:]
            if not header_written:
                f.write(header + "\n")
                header_written = True
            for line in data_lines:
                f.write(line + "\n")
            n = len(data_lines)
            total += n
            print(f"Downloaded {total} ...")
            if n < page_size:
                break
            offset += page_size
    print(f"\nSaved {total}")

In [25]:
DOWNLOAD=False
if DOWNLOAD:
    API_URL = "https://data.cityofchicago.org/resource/b8xg-w8bx.csv"
    query_params = {
        "$where": "trip_start_timestamp >= '2025-01-01T00:00:00.000' AND trip_start_timestamp < '2026-01-01T00:00:00.000'",
        "$order": "trip_start_timestamp ASC",
    }
    download_from_api(API_URL, query_params, taxi_orig_file)

Downloaded 50000 ...
Downloaded 100000 ...
Downloaded 150000 ...
Downloaded 200000 ...
Downloaded 250000 ...
Downloaded 300000 ...
Downloaded 350000 ...
Downloaded 400000 ...
Downloaded 450000 ...
Downloaded 500000 ...
Downloaded 550000 ...
Downloaded 600000 ...
Downloaded 650000 ...
Downloaded 700000 ...
Downloaded 750000 ...
Downloaded 800000 ...
Downloaded 850000 ...
Downloaded 900000 ...
Downloaded 950000 ...
Downloaded 1000000 ...
Downloaded 1050000 ...
Downloaded 1100000 ...
Downloaded 1150000 ...
Downloaded 1200000 ...
Downloaded 1250000 ...
Downloaded 1300000 ...
Downloaded 1350000 ...
Downloaded 1400000 ...
Downloaded 1450000 ...
Downloaded 1500000 ...
Downloaded 1550000 ...
Downloaded 1600000 ...
Downloaded 1650000 ...
Downloaded 1700000 ...
Downloaded 1750000 ...
Downloaded 1800000 ...
Downloaded 1850000 ...
Downloaded 1900000 ...
Downloaded 1950000 ...
Downloaded 2000000 ...
Downloaded 2050000 ...
Downloaded 2100000 ...
Downloaded 2150000 ...
Downloaded 2200000 ...
Download

## 3 Scooter

In [26]:
DOWNLOAD=False
if DOWNLOAD:
    API_URL = "https://data.cityofchicago.org/resource/2i5w-ykuw.csv"
    query_params = {
        "$where": "start_time >= '2025-01-01T00:00:00.000' AND start_time < '2026-01-01T00:00:00.000'",
        "$order": "start_time ASC",
    }
    download_from_api(API_URL, query_params, scooter_orig_file)

Downloaded 50000 ...
Downloaded 100000 ...
Downloaded 150000 ...
Downloaded 200000 ...
Downloaded 250000 ...
Downloaded 300000 ...
Downloaded 350000 ...
Downloaded 400000 ...
Downloaded 450000 ...
Downloaded 500000 ...
Downloaded 550000 ...
Downloaded 600000 ...
Downloaded 650000 ...
Downloaded 700000 ...
Downloaded 750000 ...
Downloaded 800000 ...
Downloaded 850000 ...
Downloaded 900000 ...
Downloaded 950000 ...
Downloaded 1000000 ...
Downloaded 1050000 ...
Downloaded 1100000 ...
Downloaded 1150000 ...
Downloaded 1200000 ...
Downloaded 1250000 ...
Downloaded 1300000 ...
Downloaded 1350000 ...
Downloaded 1400000 ...
Downloaded 1450000 ...
Downloaded 1500000 ...
Downloaded 1550000 ...
Downloaded 1600000 ...
Downloaded 1650000 ...
Downloaded 1700000 ...
Downloaded 1750000 ...
Downloaded 1800000 ...
Downloaded 1850000 ...
Downloaded 1900000 ...
Downloaded 1950000 ...
Downloaded 2000000 ...
Downloaded 2050000 ...
Downloaded 2100000 ...
Downloaded 2150000 ...
Downloaded 2200000 ...
Download

## 4 TNP

In [27]:
DOWNLOAD=True
if DOWNLOAD:
    API_URL = "https://data.cityofchicago.org/resource/6dvr-xwnh.csv"
    query_params = {
        "$where": "trip_start_timestamp >= '2025-01-01T00:00:00.000' AND trip_start_timestamp < '2026-01-01T00:00:00.000'",
        "$order": "trip_start_timestamp ASC",
    }
    download_from_api(API_URL, query_params, TNP_orig_file)

Downloaded 50000 ...
Downloaded 100000 ...
Downloaded 150000 ...
Downloaded 200000 ...
Downloaded 250000 ...


KeyboardInterrupt: 

## 5 Bike

In [28]:
DOWNLOAD=True
year = 2025
base_url_template = "https://divvy-tripdata.s3.amazonaws.com/{year}{month:02d}-divvy-tripdata.zip"
os.makedirs(bike_orig_dir, exist_ok=True)
if DOWNLOAD:
    for month in range(1, 13):
        url = base_url_template.format(year=year, month=month)
        zip_filename = f"{year}{month:02d}-divvy-tripdata.zip"
        zip_filepath = os.path.join(bike_orig_dir, zip_filename)
        try:
            response = requests.get(url, stream=True, timeout=30)

            if response.status_code == 200:
                with open(zip_filepath, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192 * 1024):
                        if chunk:
                            f.write(chunk)

                with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
                    zip_ref.extractall(bike_orig_dir)
                os.remove(zip_filepath)

        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")